<a href="https://colab.research.google.com/github/joshlemonte/EarthDataViz/blob/main/GBNP_GroundwaterChem_Student_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEOL 230 — Case Study 3: Great Basin Groundwater Geochemistry
## STUDENT NOTEBOOK — Starter Version

---

**Name:** ____________________________  
**Date:** ____________________________  
**Lab section:** ______________________

---

### What this notebook does
You will complete this notebook step by step to analyze groundwater chemistry from the **Basin and Range Carbonate-rock Aquifer System (BARCAS)** in eastern Nevada and western Utah. Cells marked `# TODO` require you to write or complete code.

### Deliverables from this notebook
All files should be saved to your `outputs/` folder:
- `Table1_GroundwaterChem_BARCAS.csv`
- `Figure1_SpCond_Chloride_Boxplots.png` (300 dpi)
- `Figure2_SpCond_vs_Elevation.png` (300 dpi)
- `Figure3_SiteLocationPreview.png` (300 dpi)
- `GBNP_Groundwater_GIS.csv`
- `GBNP_Climate_GIS.csv`

---
## PART 0 — Setup
Run this cell first — it imports libraries and sets your study area constants.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io, os, warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

# Study area: eastern Nevada + western Utah (BARCAS region)
BBOX        = "-117.0,37.0,-112.5,41.5"  # minLon, minLat, maxLon, maxLat
DATE_START  = "01-01-2000"
DATE_END    = "12-31-2024"
SPCOND_MAX  = 30000   # µS/cm upper QC threshold
CL_MAX      = 5000    # mg/L upper QC threshold
COLORS      = {'Well': '#4878CF', 'Spring': '#6ACC65'}

print("Setup complete.")

---
## PART 1 — Download Data from USGS Water Quality Portal

The **USGS Water Quality Portal** ([waterqualitydata.us](https://www.waterqualitydata.us)) provides a REST API that returns CSV data. You submit URL-encoded queries and get a CSV response.

We make two queries:
1. **Station query** → metadata (name, lat/lon, elevation, type)
2. **Results query** → measured values (SpCond and Cl)

The URLs are already written below. Your job is to make the HTTP requests and load the responses.

In [ ]:
# --- Station metadata download ---
print("Downloading station metadata from USGS WQP (may take ~30 sec)...")

station_url = (
    "https://www.waterqualitydata.us/data/Station/search?"
    f"bBox={BBOX}"
    "&siteType=Well&siteType=Spring"
    "&statecode=US%3A32&statecode=US%3A49"
    "&mimeType=csv&zip=no"
)

# TODO: Use requests.get() to fetch the station_url with a timeout of 120 seconds.
#       Then check the status code with raise_for_status().
#       Load the response text into a DataFrame using pd.read_csv(io.StringIO(...))
#       with dtype=str and low_memory=False. Name it stations_raw.

# YOUR CODE HERE
r_stations = ___
___
stations_raw = ___

print(f"Downloaded {len(stations_raw):,} station records.")
print(stations_raw['MonitoringLocationTypeName'].value_counts().head(8))

---
> **📘 Why this code? — REST API querying**
>
> The `requests.get(url)` + `pd.read_csv(io.StringIO(...))` pattern is how you pull data from virtually any government environmental database — USGS, EPA, NOAA, EIA, and more. Instead of downloading a file manually, you construct a URL with parameters (bounding box, date range, chemical name) and the server sends back exactly the subset you asked for. The URL you just used is a **REST API** call: the parameters after `?` are query arguments, and `&` separates them. Learning to read API documentation and adapt this structure is a skill more valuable than knowing any specific dataset — it works across disciplines and agencies.

In [ ]:
# --- Chemistry results download ---
print("Downloading chemistry results (may take 1–3 minutes)...")

result_url = (
    "https://www.waterqualitydata.us/data/Result/search?"
    f"bBox={BBOX}"
    "&characteristicName=Specific+conductance"
    "&characteristicName=Chloride"
    "&siteType=Well&siteType=Spring"
    "&statecode=US%3A32&statecode=US%3A49"
    f"&startDateLo={DATE_START}&startDateHi={DATE_END}"
    "&mimeType=csv&zip=no"
)

# TODO: Same pattern as above — fetch result_url with a timeout of 360 seconds,
#       check status, and load into results_raw.

# YOUR CODE HERE
r_results = ___
___
results_raw = ___

print(f"Downloaded {len(results_raw):,} result rows.")
print(results_raw['CharacteristicName'].value_counts())

---
## PART 2 — Clean and Process Data

The cell below selects only the columns we need and renames them. Then you will:
1. Convert coordinates and elevation to numeric (using `pd.to_numeric(..., errors='coerce')`)
2. Convert elevation from feet to meters
3. Drop rows with missing coordinates
4. Simplify site type labels to just `'Well'` or `'Spring'`

In [ ]:
# Column mapping (provided)
station_cols = {
    'MonitoringLocationIdentifier': 'site_id',
    'MonitoringLocationName':       'site_name',
    'MonitoringLocationTypeName':   'site_type',
    'LatitudeMeasure':              'latitude',
    'LongitudeMeasure':             'longitude',
    'VerticalMeasure/MeasureValue': 'elevation_ft',
    'StateCode':                    'state_code',
    'CountyCode':                   'county_code'
}
stations = stations_raw[list(station_cols.keys())].rename(columns=station_cols).copy()

# TODO: Convert latitude, longitude, and elevation_ft to numeric (use pd.to_numeric, errors='coerce')
# YOUR CODE HERE
stations['latitude']     = ___
stations['longitude']    = ___
stations['elevation_ft'] = ___

# TODO: Add a column elevation_m converting feet → meters (1 ft = 0.3048 m)
# YOUR CODE HERE
stations['elevation_m'] = ___

# TODO: Drop rows where latitude OR longitude is NaN
# YOUR CODE HERE
stations = ___

# TODO: Filter to bounding box: lon between -117.0 and -112.5, lat between 37.0 and 41.5
# YOUR CODE HERE
stations = stations[___]

# Simplify site type (provided)
stations['site_type_clean'] = stations['site_type'].apply(
    lambda x: 'Spring' if 'spring' in str(x).lower() else 'Well'
)

print(f"Clean stations: {len(stations):,} with valid coordinates")
print(stations['site_type_clean'].value_counts())

---
> **📘 Why this code? — Robust type conversion and boolean masking**
>
> Real-world data arrives with mixed types: the `LatitudeMeasure` column contains mostly numbers, but also blank strings, text flags like `"--"`, and data entry errors. `pd.to_numeric(..., errors='coerce')` converts what it can to float and silently turns everything else into `NaN` — preventing a crash. The follow-up `dropna()` removes the unusable rows. This two-step pattern (coerce → drop) appears in nearly every real environmental dataset workflow.
>
> The **bbox filter** (`stations[condition_on_lon & condition_on_lat]`) demonstrates boolean masking: a true/false test applied to every row simultaneously, keeping only rows where both conditions are true. This is how you apply spatial, temporal, or chemical quality-control thresholds in any pandas DataFrame.

In [ ]:
# Column mapping for results (provided)
result_cols = {
    'MonitoringLocationIdentifier':              'site_id',
    'CharacteristicName':                        'parameter',
    'ActivityStartDate':                         'sample_date',
    'ResultMeasureValue':                        'value_raw',
    'ResultMeasure/MeasureUnitCode':             'units',
    'ResultDetectionConditionText':              'detect_condition',
    'DetectionQuantitationLimitMeasure/MeasureValue': 'detect_limit'
}
results = results_raw[list(result_cols.keys())].rename(columns=result_cols).copy()
results['value_raw']    = pd.to_numeric(results['value_raw'],    errors='coerce')
results['detect_limit'] = pd.to_numeric(results['detect_limit'], errors='coerce')

# Handle censored values: replace with half the detection limit (provided)
censored_mask = (
    results['value_raw'].isna() &
    results['detect_condition'].str.contains(
        'Not Detected|Below|Quantification', na=False, case=False) &
    results['detect_limit'].notna()
)
results.loc[censored_mask, 'value_raw'] = results.loc[censored_mask, 'detect_limit'] * 0.5

# TODO: Drop rows still missing value_raw after censored substitution.
#       Then create column 'value' as float from value_raw.
# YOUR CODE HERE
results = ___
results['value'] = ___

print(f"Clean results: {len(results):,} rows")
print(results['parameter'].value_counts())

---
> **📘 Why this code? — Censored data and compound boolean masks**
>
> When a lab reports a value as "below detection limit," you have a **censored measurement**: you know the true value is *less than* some threshold, but not exactly what it is. The standard approach used by EPA and USGS — substituting half the detection limit — is a conservative estimate that preserves the row rather than discarding data you know something about.
>
> The three-part boolean mask (`value_raw.isna() & detect_condition.str.contains(...) & detect_limit.notna()`) selects only the rows that satisfy all three conditions simultaneously. This compound mask pattern is the general Python idiom for identifying records that meet multiple criteria — you'll use it any time you need to treat a subset of rows differently from the rest.

In [ ]:
# --- Separate and normalize parameters ---

# Specific conductance — filter and normalize to µS/cm
spcond_raw = results[results['parameter'] == 'Specific conductance'].copy()
spcond_raw.loc[spcond_raw['units'] == 'mS/cm', 'value'] *= 1000  # convert mS → µS
spcond_raw = spcond_raw[(spcond_raw['value'] > 0) & (spcond_raw['value'] < SPCOND_MAX)]

# Chloride — filter to plausible range
cl_raw = results[results['parameter'] == 'Chloride'].copy()
cl_raw = cl_raw[(cl_raw['value'] > 0) & (cl_raw['value'] < CL_MAX)]

# TODO: Aggregate spcond_raw to one median value per site_id.
#       Name the result columns 'SpCond_uScm' (median) and 'SpCond_n' (count).
#       Reset the index and call the result spcond_site.
# HINT: Use .groupby('site_id')['value'].agg(SpCond_uScm='median', SpCond_n='count').reset_index()
# YOUR CODE HERE
spcond_site = ___

# TODO: Same for cl_raw → cl_site, with columns Cl_mgL and Cl_n
# YOUR CODE HERE
cl_site = ___

print(f"Sites with SpCond: {len(spcond_site):,}")
print(f"Sites with Chloride: {len(cl_site):,}")

---
> **📘 Why this code? — GroupBy aggregation**
>
> A single monitoring well may have dozens of measurements collected over many years. To map the *site's* characteristic chemistry, you need one representative value per location. `.groupby('site_id')['value'].agg(SpCond_uScm='median', SpCond_n='count')` does this in one line: it groups all rows with the same `site_id`, then computes the median and count across those rows.
>
> The choice of **median** (rather than mean) makes the result more robust to occasional anomalous measurements — a single bad sample won't distort the result as much. The **count** column tells you how many measurements backed each median, which is important context when you're interpreting spatial patterns: a site with 1 measurement is less reliable than one with 20.
>
> This same "many measurements → one value per location" pattern appears in air quality monitoring, sediment sampling, stream gauge analysis, and any field-collected dataset.

In [ ]:
# --- Merge stations + chemistry ---
# TODO: Merge stations with spcond_site on 'site_id' using how='inner'.
#       Then merge that result with cl_site on 'site_id' using how='left'.
#       Drop duplicates on 'site_id'. Call the final result gw.
# YOUR CODE HERE
gw = ___
gw = ___
gw = ___

print(f"Final dataset: {len(gw):,} sites")
print(gw['site_type_clean'].value_counts())
print(f"SpCond range: {gw['SpCond_uScm'].min():.0f} – {gw['SpCond_uScm'].max():.0f} µS/cm")

---
> **📘 Why this code? — DataFrame merging and join types**
>
> Station metadata (name, coordinates, elevation) and chemistry results live in separate tables — this is standard database design, avoiding redundant storage. Merging on a shared key (`site_id`) reconstructs the combined record you need for analysis.
>
> The join type matters:
> - **`how='inner'`** for the first merge keeps only sites that appear in *both* tables — sites with no SpCond data are dropped. This is appropriate here because SpCond is the core variable.
> - **`how='left'`** for the second merge keeps all SpCond sites even if chloride data is missing — you don't want to lose a well just because chloride wasn't measured there.
>
> Understanding the difference between inner, left, right, and outer joins is fundamental to any multi-table data workflow — in pandas, SQL, or any database system.

---
## PART 3 — Table 1: Summary Statistics

Group by `site_type_clean` and compute: N sites, SpCond median/mean/SD/min/max, Cl median/mean/SD.
Round all values to 1 decimal place. Export to `outputs/Table1_GroundwaterChem_BARCAS.csv`.

In [ ]:
# TODO: Create summary statistics DataFrame.
#       Group by 'site_type_clean' and compute the required statistics.
#       Round to 1 decimal. Rename columns to human-readable labels.
#       Export to 'outputs/Table1_GroundwaterChem_BARCAS.csv'.

# YOUR CODE HERE
summary = (
    gw.groupby('site_type_clean')
    .agg(
        N_sites      = ('site_id',     'count'),
        # TODO: Add SpCond_med, SpCond_mean, SpCond_sd, SpCond_min, SpCond_max
        # TODO: Add Cl_med, Cl_mean, Cl_sd
    )
    .round(1)
    .reset_index()
)

# TODO: Rename columns and save to CSV

print(summary)

---
## PART 4 — Figure 1: Box Plots by Site Type

Create a **two-panel figure** with:
- Panel A: SpCond box plots (Well vs Spring)
- Panel B: Chloride box plots (Well vs Spring)

Use the colors in `COLORS`. Add a data source note. Export at 300 dpi.

The `ax.boxplot()` function takes a list of arrays, one per group. Use `patch_artist=True` to fill the boxes.

In [ ]:
# TODO: Build Figure 1 — two-panel box plots.
#       Requirements:
#         - Two side-by-side panels (Panel A: SpCond, Panel B: Chloride)
#         - Box color matches COLORS dict (Well = blue, Spring = green)
#         - Y-axis labels include units (µS/cm and mg/L)
#         - Figure title, data source note
#         - Annotate n= for each box
#         - Save to 'outputs/Figure1_SpCond_Chloride_Boxplots.png' at 300 dpi

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
site_types = ['Well', 'Spring']

# Panel A: SpCond
ax1 = axes[0]
# TODO: Extract SpCond data for each site type into a list (data_sp)
data_sp = ___
# TODO: Call ax1.boxplot() with patch_artist=True and custom median line
bp1 = ax1.boxplot(___)
# TODO: Set box colors from COLORS dict
# TODO: Set axis labels, title, remove top/right spines, add grid

# Panel B: Chloride
ax2 = axes[1]
# TODO: Same as Panel A but for Cl_mgL (only use sites with Cl data)

# TODO: Add figure title and save
plt.tight_layout()
plt.savefig('outputs/Figure1_SpCond_Chloride_Boxplots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

---
## PART 5 — Figure 2: Specific Conductance vs. Elevation

Create a scatter plot of SpCond (y-axis) vs. elevation in meters (x-axis), with points colored by site type using `COLORS`. Add a linear trend line using `np.polyfit`. Export at 300 dpi.

In [ ]:
# TODO: Build Figure 2.
#       Requirements:
#         - Scatter colored by site type (use COLORS)
#         - Linear trend line (np.polyfit degree 1, plotted as dashed black line)
#         - Report slope in the legend label (e.g., 'Trend: -2.3 µS/cm per m')
#         - Axis labels with units
#         - Save to 'outputs/Figure2_SpCond_vs_Elevation.png' at 300 dpi

# YOUR CODE HERE

print("Figure 2 saved.")

---
## PART 6 — Figure 3: Site Location Preview Map

Create a scatter plot with longitude on the x-axis and latitude on the y-axis. Color each point by `SpCond_uScm` using the `'YlOrRd'` colormap. Add a colorbar and annotate the approximate location of Great Basin NP. Export at 300 dpi.

In [ ]:
# TODO: Build Figure 3.
#       Requirements:
#         - Scatter plot using longitude/latitude as x/y
#         - Color = SpCond_uScm, cmap = 'YlOrRd'
#         - Colorbar labeled in µS/cm
#         - Annotation pointing to GBNP (~-114.25°, 38.98°)
#         - Note that this is a Python QC map
#         - Save to 'outputs/Figure3_SiteLocationPreview.png' at 300 dpi

# YOUR CODE HERE

print("Figure 3 saved.")

---
## PART 7 — Export GIS-Ready CSV

The CSV must have clean column names (no special characters), decimal-degree coordinates, and numeric values. Use the column mapping below.

In [ ]:
# TODO: Select the columns listed below, rename them, round values, and
#       export to 'outputs/GBNP_Groundwater_GIS.csv'.
#
# Columns to include (old_name → new_name):
#   site_id, site_name, site_type_clean → site_type,
#   latitude (round 6), longitude (round 6), elevation_m (round 1),
#   SpCond_uScm (round 1), SpCond_n → SpCond_n_meas,
#   Cl_mgL (round 2), Cl_n → Cl_n_meas, state_code → state

# YOUR CODE HERE

print("GIS CSV exported.")

---
## PART 8 — NOAA Climate Normals

The cell below loads pre-processed 1991–2020 NOAA climate normals for 25 stations in the BARCAS region. Once loaded, create Figure 4 and export the climate GIS CSV.

In [ ]:
# --- NOAA 1991-2020 climate normals for the BARCAS region ---
# Data pre-processed from NOAA National Centers for Environmental Information (NCEI)
# 25 GHCN stations in eastern Nevada and western Utah

CLIMATE_CSV = """\
station_id,station_name,latitude,longitude,elevation_m,annual_precip_mm,mean_ann_temp_C,normals_period,source
USC00260050,BAKER NV,38.508,-114.133,1552,208.6,9.4,1991-2020,NOAA NCEI
USC00260255,BEOWAWE NV,40.596,-116.488,1436,211.8,9.8,1991-2020,NOAA NCEI
USW00023169,CALIENTE NV,37.617,-114.517,1097,147.0,12.2,1991-2020,NOAA NCEI
USC00260878,CHERRY CREEK NV,40.165,-114.888,1854,263.8,6.8,1991-2020,NOAA NCEI
USW00023179,ELY NV WSO AP,39.3,-114.842,1906,256.5,7.0,1991-2020,NOAA NCEI
USC00262243,EUREKA NV,39.513,-115.957,1993,274.1,6.7,1991-2020,NOAA NCEI
USC00263126,HAMILTON NV,39.241,-115.544,2012,279.2,6.3,1991-2020,NOAA NCEI
USC00263680,LAGES STATION NV,41.0,-114.7,1720,241.5,7.1,1991-2020,NOAA NCEI
USC00264220,MCGILL NV,39.403,-114.782,1847,255.8,7.8,1991-2020,NOAA NCEI
USC00264355,MINOR NV,37.883,-116.35,1585,209.8,9.4,1991-2020,NOAA NCEI
USC00265171,PIOCHE NV,37.933,-114.45,1889,252.5,6.9,1991-2020,NOAA NCEI
USC00265511,RAILROAD VALLEY NV,38.467,-115.683,1502,200.2,10.0,1991-2020,NOAA NCEI
USC00265821,SCHELLBOURNE NV,39.608,-114.658,1780,256.9,7.5,1991-2020,NOAA NCEI
USC00266931,WARM SPRINGS NV,38.183,-116.433,1567,214.0,9.6,1991-2020,NOAA NCEI
USC00267340,WHITE RIVER NARROWS NV,38.7,-115.0,1704,239.5,8.0,1991-2020,NOAA NCEI
USC00420512,BEAVER UT,38.275,-112.634,1794,244.3,7.4,1991-2020,NOAA NCEI
USC00421263,CEDAR CITY UT AP,37.701,-113.099,1712,191.6,8.5,1991-2020,NOAA NCEI
USC00421831,DELTA UT,39.352,-112.577,1448,193.6,10.2,1991-2020,NOAA NCEI
USC00423323,IBAPAH UT,39.633,-113.983,1546,226.6,8.6,1991-2020,NOAA NCEI
USC00424856,MILFORD UT,38.4,-113.017,1534,217.4,9.2,1991-2020,NOAA NCEI
USC00425612,OSCEOLA UT,39.9,-113.183,1615,228.0,8.5,1991-2020,NOAA NCEI
USC00426001,PARTOUN UT,39.95,-113.717,1527,220.2,9.5,1991-2020,NOAA NCEI
USC00427176,SNAKE VALLEY UT,39.083,-114.083,1536,220.6,9.5,1991-2020,NOAA NCEI
USC00427236,SOUTHERN UT UNIV,37.677,-113.062,1745,234.6,8.7,1991-2020,NOAA NCEI
USC00427924,WAH WAH VALLEY UT,38.6,-113.35,1645,214.7,8.6,1991-2020,NOAA NCEI"""

climate_df = pd.read_csv(io.StringIO(CLIMATE_CSV))
print(f"Loaded {len(climate_df)} NOAA climate stations")
print(f"Elevation range: {climate_df['elevation_m'].min():.0f}–{climate_df['elevation_m'].max():.0f} m")
print(f"Precip range: {climate_df['annual_precip_mm'].min():.1f}–{climate_df['annual_precip_mm'].max():.1f} mm/yr")
climate_df.head()

---
> **📘 Why this code? — Pre-processed data and in-memory file objects**
>
> The climate data here comes embedded directly in the notebook as a text string, then loaded with `pd.read_csv(io.StringIO(...))`. `io.StringIO` creates an **in-memory file object** — it lets `read_csv` treat a plain string exactly as if it were a file on disk, with no actual file needed. You'll use this pattern any time you want to pass structured text data to a function that normally expects a filename.
>
> Why not download live from NOAA? The standard tool for this is the `meteostat` library, which wraps the NOAA GHCN database in a clean Python API. However, `meteostat` currently has compatibility issues with the versions of NumPy and pandas that Colab uses, so we pre-downloaded the relevant stations and embedded them here. This illustrates an important real-world pattern: **sometimes a data pipeline fails, and the pragmatic fix is to cache the data and move on** — especially when the data itself (1991–2020 normals) doesn't change.

In [ ]:
# TODO: (A) Create Figure 4: scatter of annual_precip_mm (y) vs elevation_m (x)
#           with a linear trend line. Save to 'outputs/Figure4_Precip_vs_Elevation.png'.
#
#       (B) Export climate_df to 'outputs/GBNP_Climate_GIS.csv'.

# YOUR CODE HERE

print("Figure 4 and climate CSV complete.")

---
## PART 9 — Coding Reflection

**Question (include your answer in your written report):**

Choose **one** coding technique from Parts 1–8 that you believe is transferable beyond this specific assignment. In 3–5 sentences:

1. **Name and explain the technique** — what does it do, in plain language?
2. **Describe a new scenario** — a different dataset, question, or discipline where you would use the same technique
3. **Explain the transfer** — what makes it applicable across contexts?

*Examples of techniques to consider: REST API querying, `pd.to_numeric` with `errors='coerce'`, GroupBy aggregation, join types in DataFrame merging, censored data handling, `try/except` in loops.*

Write your 3–5 sentence answer below (double-click this cell to edit):

---

*(Your reflection here)*

---
## Final Checklist

- [ ] `Table1_GroundwaterChem_BARCAS.csv`
- [ ] `Figure1_SpCond_Chloride_Boxplots.png` (300 dpi)
- [ ] `Figure2_SpCond_vs_Elevation.png` (300 dpi)
- [ ] `Figure3_SiteLocationPreview.png` (300 dpi)
- [ ] `Figure4_Precip_vs_Elevation.png` (300 dpi)
- [ ] `GBNP_Groundwater_GIS.csv`
- [ ] `GBNP_Climate_GIS.csv`

---
**AI Use Log** (required)

| Step | Tool used | What it helped with | How I verified the output |
|------|-----------|--------------------|--------------------------|
| | | | |